In [13]:
import re
from collections import Counter
import numpy as np

def tokenize(text: str):
    text = text.lower()
    return re.findall(r"[a-z0-9']+", text)

def build_vocab(tokens, vocab_size=30000, min_count=5):
    counts = Counter(tokens)
    items = [(w, c) for w, c in counts.items() if c >= min_count]
    items.sort(key=lambda x: x[1], reverse=True)

    items = items[: max(0, vocab_size - 1)]

    word2id = {"<unk>": 0}
    id2word = ["<unk>"]

    for w, _ in items:
        word2id[w] = len(id2word)
        id2word.append(w)

    return word2id, id2word, counts

def encode(tokens, word2id):
    return [word2id.get(t, 0) for t in tokens]

def sample_skipgram_pairs(corpus_ids, batch_size=1024, window_size=2, rng=None, dynamic_window=False):
    """
    Returns:
      centers: (B,) int64
      contexts: (B,) int64
    """
    if rng is None:
        rng = np.random.default_rng()

    corpus = np.asarray(corpus_ids, dtype=np.int64)
    N = corpus.shape[0]

    idx = rng.integers(low=window_size, high=N - window_size, size=batch_size)
    centers = corpus[idx]

    if not dynamic_window:
        offsets = np.arange(-window_size, window_size + 1)
        offsets = offsets[offsets != 0]
        chosen_offsets = rng.choice(offsets, size=batch_size, replace=True)
    else:
        chosen_offsets = np.empty(batch_size, dtype=np.int64)

        for i in range(batch_size):
            current_window = rng.integers(1, window_size + 1)
            offsets = np.arange(-current_window, current_window + 1)
            offsets = offsets[offsets != 0]
            chosen_offsets[i] = rng.choice(offsets)

    contexts = corpus[idx + chosen_offsets]
    return centers, contexts

def build_negative_sampling_dist(counts, word2id):
    vocab_size = len(word2id)
    freqs = np.zeros(vocab_size)

    for word, idx in word2id.items():
        freqs[idx] = counts.get(word, 0)

    freqs = freqs ** 0.75
    dist = freqs / freqs.sum()

    return dist

def sample_negative_words_filtered(dist, contexts, K, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    B = len(contexts)
    vocab_size = len(dist)

    negatives = np.empty((B, K), dtype=np.int64)

    for i in range(B):
        count = 0
        while count < K:
            w = rng.choice(vocab_size, p=dist)

            if w != contexts[i]:   
                negatives[i, count] = w
                count += 1

    return negatives

def sigmoid(x):
    x = np.clip(x, -50, 50)
    return 1 / (1 + np.exp(-x))

def init_embeddings(vocab_size, embed_dim=100, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    W_in = rng.normal(0, 0.01, size=(vocab_size, embed_dim))
    W_out = rng.normal(0, 0.01, size=(vocab_size, embed_dim))

    return W_in, W_out

def train_step(W_in, W_out, centers, contexts, negatives, lr=0.025):
    """
    centers:   (B,)
    contexts:  (B,)
    negatives: (B, K)
    """

    v = W_in[centers]         
    u_pos = W_out[contexts]   
    u_neg = W_out[negatives]  

    s_pos = np.sum(v * u_pos, axis=1)                 
    s_neg = np.sum(u_neg * v[:, None, :], axis=2)     

    pos_loss = np.logaddexp(0, -s_pos)
    neg_loss = np.sum(np.logaddexp(0, s_neg), axis=1)
    loss = np.mean(pos_loss + neg_loss)

    g_pos = sigmoid(s_pos) - 1.0      
    g_neg = sigmoid(s_neg)            

    grad_v = g_pos[:, None] * u_pos + np.sum(g_neg[:, :, None] * u_neg, axis=1)  

    grad_u_pos = g_pos[:, None] * v   

    grad_u_neg = g_neg[:, :, None] * v[:, None, :] 

    np.add.at(W_in, centers, -lr * grad_v)

    np.add.at(W_out, contexts, -lr * grad_u_pos)

    B, K = negatives.shape
    np.add.at(W_out, negatives.reshape(B * K), -lr * grad_u_neg.reshape(B * K, -1))

    return loss

def build_subsampling_probs(counts, word2id, total_tokens, t=1e-5):
    keep_probs = np.ones(len(word2id), dtype=np.float64)

    for word, idx in word2id.items():
        if idx == 0:  # <unk>
            continue

        freq = counts[word] / total_tokens
        keep_probs[idx] = min(1.0, np.sqrt(t / freq) + (t / freq))

    return keep_probs

def subsample_corpus(corpus_ids, keep_probs, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    corpus = np.asarray(corpus_ids, dtype=np.int64)
    mask = rng.random(len(corpus)) < keep_probs[corpus]
    return corpus[mask]


text = open("data/text.txt", "r", encoding="utf-8").read()
tokens = tokenize(text)
word2id, id2word, counts = build_vocab(tokens, vocab_size=20000, min_count=5)

corpus_ids = encode(tokens, word2id)

rng = np.random.default_rng(42)

keep_probs = build_subsampling_probs(
    counts=counts,
    word2id=word2id,
    total_tokens=len(tokens),
    t=1e-5
)

corpus_ids = subsample_corpus(corpus_ids, keep_probs, rng=rng)
print("original corpus size:", len(tokens))
print("subsampled corpus size:", len(corpus_ids))

dist = build_negative_sampling_dist(counts, word2id)

W_in, W_out = init_embeddings(len(word2id), embed_dim=50, rng=rng) # 50, 100, 200

rng = np.random.default_rng(42)

B = 256
K = 5
steps = 2000
lr = 0.05 # 0.05 0.025 0.01
ws = 5 # 2 3 5

for step in range(steps):
    current_lr = lr * max(0.1, 1 - step / steps)
    centers, contexts = sample_skipgram_pairs(
        corpus_ids,
        batch_size=B,
        window_size=ws,
        rng=rng,
        dynamic_window=True
    )
    # negatives = sample_negative_words(dist, B * K, rng=rng).reshape(B, K)
    negatives = sample_negative_words_filtered(dist, contexts, K, rng)

    loss = train_step(W_in, W_out, centers, contexts, negatives, lr=current_lr)

    if (step+1) % 100 == 0 or step == 0:
        print(f"step={step}, loss={loss:.4f}")

# loss =  2.6324 (simmilar for embedded dimension 100 and 200)

original corpus size: 17005207
subsampled corpus size: 5882477
step=0, loss=4.1589
step=99, loss=3.8579
step=199, loss=3.6349
step=299, loss=3.4270
step=399, loss=3.3213
step=499, loss=3.1948
step=599, loss=3.1983
step=699, loss=3.0625
step=799, loss=2.9274
step=899, loss=2.8493
step=999, loss=2.9731
step=1099, loss=2.8361
step=1199, loss=2.8377
step=1299, loss=2.7088
step=1399, loss=2.7622
step=1499, loss=2.7066
step=1599, loss=2.7637
step=1699, loss=2.7280
step=1799, loss=2.7951
step=1899, loss=2.6703
step=1999, loss=2.6324
